# Titanic Survival Classification — ML Foundations Refresh

Goal: predict passenger survival, and produce a full evaluation writeup using precision, recall, F1, and ROC-AUC.

This is a refresher exercise, not a portfolio project — the point is nailing the *process*: clean train/test split, cross-validation, picking the right metrics, and honestly interpreting results.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub
from kagglehub import KaggleDatasetAdapter

sns.set_theme(style="whitegrid")

Matplotlib is building the font cache; this may take a moment.


## 1. Load the data

In [3]:
# Download Titanic dataset via kagglehub
path = kagglehub.dataset_download("yasserh/titanic-dataset")
print("Dataset downloaded to:", path)

import os
print(os.listdir(path))

Dataset downloaded to: C:\Users\farha\.cache\kagglehub\datasets\yasserh\titanic-dataset\versions\1
['Titanic-Dataset.csv']


In [4]:
# Adjust filename below to match whatever kagglehub prints above
df = pd.read_csv(os.path.join(path, "Titanic-Dataset.csv"))
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Explore the data
Before touching any model: understand what you're working with. Shape, missing values, target balance.

In [5]:
print(df.shape)
df.info()

(891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
# Missing values
df.isnull().sum().sort_values(ascending=False)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

In [7]:
# Target balance — is this dataset imbalanced? (relevant to why we're not just using accuracy)
df["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

## 3. Clean + prepare features

Stop here and think before running the next cell — what's your plan for:
- Missing `Age` values?
- The `Cabin` column (lots of missing values)?
- Categorical columns like `Sex` and `Embarked` — how will you convert them to numbers?

In [8]:
# TODO: handle missing values and encode categorical columns
# Come back to this after deciding your approach above
# Drop columns we won't use: PassengerId (no signal), Name/Ticket (raw text, too complex for now), Cabin (replaced below)
df["has_cabin"] = df["Cabin"].notnull().astype(int)
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Age: median imputation
df["Age"] = df["Age"].fillna(df["Age"].median())

# Embarked: drop the 2 missing rows
df = df.dropna(subset=["Embarked"])

# Sex: map to 0/1
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

# Embarked: one-hot encode
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)  # drop_first avoids redundant column

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,has_cabin,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,0,False,True
1,1,1,1,38.0,1,0,71.2833,1,False,False
2,1,3,1,26.0,0,0,7.9250,0,False,True
3,1,1,1,35.0,1,0,53.1000,1,False,True
4,0,3,0,35.0,0,0,8.0500,0,False,True


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # probability of "survived"

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

[[97 13]
 [19 49]]
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.79      0.72      0.75        68

    accuracy                           0.82       178
   macro avg       0.81      0.80      0.81       178
weighted avg       0.82      0.82      0.82       178

ROC-AUC: 0.8612299465240643


In [10]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
print("CV ROC-AUC scores:", cv_scores)
print("Mean:", cv_scores.mean(), "Std:", cv_scores.std())

CV ROC-AUC scores: [0.84719251 0.83081551 0.8559492  0.83602941 0.87540475]
Mean: 0.8490782760143258 Std: 0.015796521158395155


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_proba))

rf_cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring="roc_auc")
print("CV ROC-AUC scores:", rf_cv_scores)
print("Mean:", rf_cv_scores.mean(), "Std:", rf_cv_scores.std())

[[93 17]
 [19 49]]
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       110
           1       0.74      0.72      0.73        68

    accuracy                           0.80       178
   macro avg       0.79      0.78      0.78       178
weighted avg       0.80      0.80      0.80       178

ROC-AUC: 0.8405080213903743
CV ROC-AUC scores: [0.84291444 0.80294118 0.9026738  0.84144385 0.90764976]
Mean: 0.8595246038365305 Std: 0.03995592253642277


In [12]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

Sex           0.270861
Fare          0.250567
Age           0.239674
Pclass        0.072120
SibSp         0.048865
has_cabin     0.044275
Parch         0.036543
Embarked_S    0.023394
Embarked_Q    0.013700
dtype: float64
